# The 2D Wavelength Solution

The book has produced two calibration products and applied them one after the other: the **tilt
solution** of [Chapter 6](06-tilt-correction.ipynb), mapping detector pixels into a rectified space
where a column is one wavelength, and the **1D wavelength solution** of
[Chapter 10](10-wavelength-solution-interactive.ipynb) in that rectified space. Chained, they give
the wavelength of *any* pixel on the raw detector, and `specreduce.wavesol2d.WavelengthSolution2D`
is that chain as an object. The chapter takes three consequences in turn.

- **A wavelength solution for the whole frame.** `pix_to_wav(disp, cdisp)` gives the wavelength of
  any detector pixel, and `wav_to_pix` inverts it, with no rectification and no assumption that a
  column is one wavelength.
- **Rectification and calibration in one step.** `resample` takes a raw frame straight to a
  wavelength grid, so background subtraction and extraction can be done in wavelength space.
- **A calibration that travels.** The combined solution is exposed as a GWCS object and serialized
  to ASDF with the chained mapping as a standalone GWCS, so code that is not `specreduce` can
  evaluate it. That is the main reason the object exists: it lets you bring your own extraction,
  including one that works on the raw detector pixels without resampling, and still have the
  wavelength of every pixel.

In [ ]:
import sys

sys.path.append("../src")

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import simple_norm
from specreduce.tilt_solution import TiltSolution
from specreduce.wavesol1d import WavelengthSolution1D
from specreduce.wavesol2d import WavelengthSolution2D

from common import (
    TILT_SOLUTION_FILE,
    WAVELENGTH_SOLUTION_FILE,
    fit_tilt_solution,
    fit_wavelength_solution,
    read_data,
)

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
ny, nx = obj.data.shape

## Combining the two solutions

Both components are on disk as a few kilobytes of analytic model each, written by Chapters 6 and
10, so the combination reads them back and refits nothing.

In [ ]:
if not TILT_SOLUTION_FILE.exists():  # normally written by Chapter 6
    fit_tilt_solution(arcs).to_asdf(TILT_SOLUTION_FILE)
tilt = TiltSolution.from_asdf(TILT_SOLUTION_FILE)

if not WAVELENGTH_SOLUTION_FILE.exists():  # normally written by Chapter 10
    fit_wavelength_solution(arcs, tilt).to_asdf(WAVELENGTH_SOLUTION_FILE)
wavelength = WavelengthSolution1D.from_asdf(WAVELENGTH_SOLUTION_FILE)

solution = WavelengthSolution2D(tilt, wavelength)

print(f"image shape   {solution.image_shape}")
print(f"dispersion    axis {solution.disp_axis}")
print(f"unit          {solution.unit}, air wavelengths: {solution.wave_air}")
print(f"pixel bounds  {solution.bounds_pix}")

That is the whole construction. The cross-dispersion coordinate passes through the tilt
transformation unchanged, so the chain is `(disp, cdisp) → rectified column → wavelength`. The one
thing `WavelengthSolution2D` needs beyond its components is the **image shape**, because the
inverse tilt transformation is fitted numerically over the frame. Here it comes from the tilt
solution's stored bounding box; pass `image_shape=(ny, nx)` if yours has none.

## What wavelength is this pixel?

The question now has an answer anywhere on the detector, not only in a rectified frame.

In [ ]:
column = 512
rows = np.array([10, 70, 135, 250, 350])
wavelengths = np.array([float(solution.pix_to_wav(column, row)) for row in rows])

print(f"wavelength down column {column} of the raw detector:")
for row, value in zip(rows, wavelengths):
    print(f"  row {row:3d}:  {value:9.3f} A")
print(f"\nspread across the illuminated slit: {np.ptp(wavelengths):.2f} A")
print(f"in pixels, at this dispersion:      {np.ptp(wavelengths) / 5.29:.2f} pix")

A single column of the raw detector spans **31.5 Å** between the ends of the illuminated slit,
about 6 pixels and more than 50 times the 0.57 Å rms of the wavelength fit: the tilt of Chapter 6
in physical units. The inverse mapping is the more useful one in practice: given a wavelength and
a position along the slit, where on the detector does it fall?

In [ ]:
line = 7635.106  # a bright ArI line from the GTC list

print(f"where {line} A lands, row by row:")
columns = np.array([float(solution.wav_to_pix(line, row)) for row in rows])
for row, value in zip(rows, columns):
    print(f"  row {row:3d}:  column {value:8.3f}")
print(f"\nspread: {np.ptp(columns):.2f} pixels")

The same 6 pixels, seen from the other side: the shape of the arc lines in Chapter 5's figure, now
a function you can evaluate. `wav_to_pix` returns `nan` for wavelengths the solution does not
cover, and both methods accept masked arrays and propagate the mask, which suits coordinates from a
detection list that already carries one.

## Resampling the whole frame

`resample` bins every row of a raw frame onto a common wavelength grid. Each row's bin edges are
mapped back through the wavelength solution and the tilt transformation into that row's detector
columns, and the enclosed flux is summed with fractional weights at the edges, so rectification
and wavelength calibration become one operation.

In [ ]:
calibrated = solution.resample(obj)

print(f"type              {type(calibrated).__name__}")
print(
    f"shape             {calibrated.flux.shape}, spectral axis index {calibrated.spectral_axis_index}"
)
print(f"flux unit         {calibrated.flux.unit}")
print(
    f"spectral axis     {calibrated.spectral_axis[0]:.1f} to {calibrated.spectral_axis[-1]:.1f}"
)
print(f"uncertainty       {type(calibrated.uncertainty).__name__}")
print(f"mask              {int(np.sum(calibrated.mask))} bins flagged")

The result is a 2D `Spectrum` with a wavelength spectral axis, a propagated variance, and a mask
flagging any bin that drew on a masked detector pixel. Its flux is a **density** in
`electron / Angstrom`, as with `WavelengthSolution1D.resample` in Chapter 10, so the conserved
quantity is the integral over wavelength, not the sum of the values.

In [ ]:
edges = calibrated.spectral_axis.bin_edges.to(u.AA).value
integral = np.nansum(np.nan_to_num(calibrated.flux.value) * np.diff(edges)[None, :])

print(f"integral over wavelength: {integral:.6e} electron")
print(f"raw detector total:       {obj.data.sum():.6e} electron")
print(f"ratio:                    {integral / obj.data.sum():.6f}")

Flux is conserved to three parts in ten thousand, the same edge loss as the tilt correction's in
Chapter 6: light at the corners of the frame, where the most strongly tilted rows run past the
reference row's coverage. The figure shows the raw detector above the calibrated frame, with
**iso-wavelength contours** on the raw panel, each tracing one wavelength across the detector with
`wav_to_pix` at every row.

In [ ]:
fig, axs = plt.subplots(2, figsize=(8, 5.0), constrained_layout=True)

contour_wavelengths = np.arange(5500.0, 10001.0, 500.0)
rows_all = np.arange(float(ny))

axs[0].imshow(
    obj.data,
    origin="lower",
    aspect="auto",
    cmap=plt.cm.Blues,
    norm=simple_norm(obj.data, stretch="log", vmin=0, vmax=250_000),
    extent=(0, nx - 1, 0, ny - 1),
)
for value in contour_wavelengths:
    columns = np.asarray(solution.wav_to_pix(value, rows_all), dtype=float)
    axs[0].plot(columns, rows_all, c="C1", lw=0.8, alpha=0.75)
    axs[0].text(
        np.nanmedian(columns),
        ny - 40,
        f"{value:.0f}",
        color="C1",
        size=6,
        ha="center",
        va="top",
        bbox=dict(fc="white", ec="none", alpha=0.6, pad=0.8),
    )
axs[0].set(
    xlim=(0, nx - 1),
    ylim=(0, ny - 1),
    xlabel="Dispersion [pix]",
    ylabel="Cross-dispersion [pix]",
)
axs[0].text(
    0.01,
    0.06,
    "raw detector, iso-wavelength contours [$\\AA$]",
    transform=axs[0].transAxes,
    va="bottom",
    size=8,
)

axis = calibrated.spectral_axis.to(u.AA).value
panel = np.nan_to_num(calibrated.flux.value)
axs[1].imshow(
    panel,
    origin="lower",
    aspect="auto",
    cmap=plt.cm.Blues,
    norm=simple_norm(panel, stretch="log", vmin=0, vmax=250_000 / 5.29),
    extent=(axis[0], axis[-1], 0, ny - 1),
)
for value in contour_wavelengths:
    axs[1].axvline(value, c="C1", lw=0.8, alpha=0.75)
axs[1].set(
    xlim=(axis[0], axis[-1]),
    ylim=(0, ny - 1),
    xlabel=r"Wavelength [$\AA$]",
    ylabel="Cross-dispersion [pix]",
)
axs[1].text(
    0.01,
    0.06,
    "wavelength calibrated, same wavelengths",
    transform=axs[1].transAxes,
    va="bottom",
    size=8,
);

The same ten wavelengths are curves on the detector and straight vertical lines after resampling,
which is what the operation does. The contours **bow as well as lean**, because the tilt is a 2D
polynomial rather than a shear, which is why Chapter 6 needed a surface fit rather than a single
angle. The lean also **changes with wavelength**: the 5500 Å contour spans about 7.6 pixels across
the slit and the 10000 Å contour about 5.7, so a correction fitted at one end of the range would
leave a residual of nearly 2 pixels at the other.

## Taking the solution elsewhere

`resample` allows a pipeline order the book has not used: calibrating the 2D frame first and
extracting in wavelength space, instead of rectifying, extracting, and calibrating the 1D spectrum.
On these data the two orderings give the same spectrum to within two parts in 100 000 across the
range, with no trend. That is the end-to-end check on the whole chain: had the tilt, the
wavelength solution or the chained resampling been wrong anywhere, the two would have parted
company where the tilt is largest. The ordering is therefore a matter of convenience. Calibrating
first resamples once where the book's order resamples twice, but neither extracts from independent
pixels, since both work on a resampled frame.

The only extraction that keeps the detector's own statistics runs on the raw frame, and that is the
case the combined solution is built for. Such a routine needs only the wavelength of each pixel it
touches, and the solution supplies it to code outside `specreduce` in two forms. In memory, the
`gwcs` property is the chained detector-pixel-to-wavelength mapping as a GWCS object. On disk,
`to_asdf` writes the tilt and wavelength solutions separately, which `from_asdf` reads back, and
the chained mapping once more as a standalone GWCS that any ASDF-aware tool can evaluate.

In [ ]:
combined_file = WAVELENGTH_SOLUTION_FILE.with_name("wavelength_solution_2d.asdf")
solution.to_asdf(combined_file)
restored = WavelengthSolution2D.from_asdf(combined_file)

# The same mapping as a GWCS object, for code that is not specreduce.
wcs = solution.gwcs
print(f"gwcs frames:  {wcs.input_frame.name} -> {wcs.output_frame.name}")
print(f"gwcs(512, 135) = {float(wcs(512, 135)):.3f} {solution.unit}")

## Summary

- **`WavelengthSolution2D` chains a tilt solution with a 1D wavelength solution** into the
  wavelength of any pixel on the raw detector. It is built from the two fitted objects, read back
  from their ASDF files, and needs only the image shape besides, to fit the inverse tilt
  transformation.
- `pix_to_wav` and `wav_to_pix` answer "what wavelength is this pixel?" and its inverse anywhere on
  the frame. A single detector column spans **31.5 Å**, about 6 pixels, between the ends of the
  illuminated slit. Both accept masked arrays, and `wav_to_pix` returns `nan` outside the
  solution's coverage.
- `resample` takes a raw frame to a wavelength-calibrated 2D `Spectrum` in one step, with the
  variance propagated, a mask for bins fed by masked pixels, and the flux as a **density** whose
  integral over wavelength is conserved to three parts in ten thousand.
- **The iso-wavelength contours** are the most direct picture of the distortion. They bow as well
  as lean, and the lean changes with wavelength, from about 7.6 pixels across the slit at 5500 Å
  to 5.7 at 10000 Å.
- **Pipeline order is a matter of convenience.** Calibrating the 1D spectrum or calibrating the 2D
  frame first give the same spectrum to two parts in 100 000, the end-to-end check on the whole
  chain. The second resamples once instead of twice, but both extract from a resampled frame.
- **The solution is built to travel.** Only extraction on the raw detector keeps independent pixel
  statistics, and such a routine needs the wavelength of every raw pixel. The `gwcs` property
  supplies it in memory, and `to_asdf` stores the components plus the chained mapping as a
  standalone GWCS, so the solution can be evaluated without `specreduce`.

Next: [Flux Calibration](13-flux-calibration.ipynb), the one calibration step this book does not
yet perform.